# MOTIF: Multi-Omics Transformer for Inferring Function

Interactive, cell-by-cell version of the pipeline you can also run from the command line with `python main.py --mode all`.

This notebook is a thin wrapper: each cell below calls the same functions used by `main.py`, so the actual model/training code lives in one place (`src/`, `data/`) instead of being duplicated here. That keeps the codebase maintainable while still giving a point-and-click, shareable entry point for Teams.

**Pipeline stages:** download data -> preprocess -> self-supervised pretraining -> fine-tuning + baselines -> diffusion training -> evaluation.

**Note:** training runs on CPU by default and can take a while (pretraining alone can run 20-40+ minutes depending on the epoch budget in `src/configs/config.py`). Each stage below can be run independently once its inputs exist — you don't have to re-run earlier cells every time.

## Setup

Run this first. It makes sure the notebook's working directory is the project root (not `notebooks/`), since the pipeline code uses paths like `data/raw` and `outputs/checkpoints` relative to the project root.

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Working directory:", Path.cwd())

*(Optional, first time only)* Install dependencies:

In [ ]:
# !pip install -r requirements.txt

## Step 1: Download Data

Pulls real gene expression data from GTEx and real chromatin accessibility (ATAC-seq) + DNA methylation (WGBS) data from ENCODE. Files already downloaded are skipped automatically. See `data/download_data.py` for the exact accessions used.

In [ ]:
from data.download_data import download_gtex, download_encode, download_methylation

download_gtex()
download_encode()
download_methylation()

## Step 2: Preprocess Data

Three sub-steps:
1. Condense the real ENCODE ATAC-seq peaks into a per-gene, per-tissue accessibility profile.
2. Condense the real ENCODE WGBS methylation calls into a per-gene, per-tissue methylation profile.
3. Build the final training dataset: real GTEx expression for real donors, combined with the real ATAC/methylation tissue-level signal from steps 1-2, split into train/val/test.

In [ ]:
from data.preprocess_atac import build_real_atac_profile
from data.preprocess_methylation import build_real_methylation_profile
from data.preprocess import build_dataset

build_real_atac_profile()
build_real_methylation_profile()
build_dataset()

## Step 3: Self-Supervised Pretraining

Trains MOTIF (the multi-omics transformer) with a masked-reconstruction objective — no expression labels used yet, just learning structure from the data itself. Saves the best checkpoint to `outputs/checkpoints/pretrained_best.pt`.

In [ ]:
from src.training.pretrain import pretrain

pretrain()

## Step 4: Fine-Tuning + Baseline Comparison

Trains four models on the supervised expression-prediction task: linear regression, an MLP, a transformer trained from scratch, and the self-supervised pretrained MOTIF encoder from Step 3. The comparison against the simpler baselines is what tells us whether MOTIF is learning something meaningful rather than just adding parameters.

In [ ]:
from src.training.finetune import finetune

finetune()

## Step 5: Train the Conditional Diffusion Model

Trains a DDPM (denoising diffusion probabilistic model) to generate new, tissue-conditioned synthetic multi-omics profiles.

In [ ]:
from src.training.train_diffusion import train_diffusion

train_diffusion()

## Step 6: Full Evaluation

Runs same-tissue expression prediction, cross-tissue generalization testing, cross-tissue transfer experiments, diffusion quality checks, and biological validation. Generates all figures, an interactive HTML dashboard, and a journal-style manuscript draft — all saved under `outputs/`.

In [ ]:
from src.evaluation.evaluate import run_evaluation

run_evaluation()

## Step 7: View Results

A few quick ways to look at what the pipeline produced without leaving the notebook.

In [ ]:
from IPython.display import Image, display, Markdown

# Key result figures
display(Image(filename="outputs/figures/1_model_comparison.png"))
display(Image(filename="outputs/figures/2_cross_tissue_heatmap.png"))

In [ ]:
# Full manuscript draft, rendered inline
display(Markdown(Path("outputs/results/manuscript_draft.md").read_text(encoding="utf-8")))

---

**Sharing this notebook / project folder:** exclude `venv/` (rebuild it from `requirements.txt`) and `data/raw/` (the real downloaded GTEx/ENCODE files are several GB — regenerate locally with `data/download_data.py` instead of uploading them). Everything else, including this notebook, `outputs/`, and `docs/`, is small and safe to share. See the root `README.md` for the full project structure.